# 08 - Monitoring, Explainability, Security, and Deployment Preparation

## Definition
Production readiness includes observability, explainability, and deployment controls.

## Theory
Good packaged model is not only accurate; it is measurable, explainable, and recoverable.

## Motivation
Operations teams need metrics, logs, and secure artifact handling to trust ML services.


In [ ]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [ ]:
from ml_package.logging_config import PredictionLogger

logger = PredictionLogger("notebook08.predictions")
logger.log_prediction(
    features={"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
    prediction=0,
    confidence=0.98,
    latency_ms=2.45,
    model_version="v2",
)
logger.log_error([None, None, None, None], "validation_failed")
logger.get_stats()


## Explainable AI with SHAP

### Global explanation
Feature importance across reference set.

### Local explanation
Contribution for one sample prediction.


In [ ]:
import numpy as np
from ml_package.model_loader import ModelLoader
from ml_package.explainability import ModelExplainer

model = ModelLoader("models/iris_model.pkl", verify_integrity=True).load()
background = np.load("models/background_data.npy")
explainer = ModelExplainer(model, background)

local = explainer.explain_single(np.array([[5.1, 3.5, 1.4, 0.2]], dtype=float))
global_imp = explainer.get_global_importance(background)

print("Local keys:", list(local.keys()))
print("Global keys:", list(global_imp.keys()))


## Security Considerations Checklist
- Unsafe deserialization risk for pickle/joblib
- Require checksum manifest in production
- Allow-list trusted digests
- Validate all API payloads with Pydantic
- Avoid direct artifact loading from untrusted sources


## Deployment Preparation

### Docker
Containerize API with immutable dependencies.

### Make Targets
Standardized commands for train/test/api/notebooks.

### Monitoring Endpoints
- JSON metrics: `/metrics`
- Prometheus metrics: `/metrics/prometheus`
